## SRP184741

**paper:** [Vertebrate Genomes Project](https://genome10k.ucsc.edu/) - lots of papers, will include as appropriate

**date, curator:** 2026-04-15, Sara Carsanaro

**notes**
* lots of samples to remove
    * pacbio/oxford nanopore samples - 71 libraries
    * cell culture samples (see rejection file for papers confirming cell culture) - 3 libraries
    * i removed one library that was marked as male but tissue was ovary, could not confirm tissue
* associated papers PMID: 33305470, PMID: 39786563, PMID: 36749728, PMID: 37141262, PMID: 37590950, PMID: 38513109, PMID: 40756765, PMID: 36662619, https://www.biorxiv.org/content/10.1101/2025.09.25.678567v1
    * Archocentrus centrarchus - PMID: 33305470
    * Alosa sapidissima - PMID: 39786563
    * Dermochelys coriacea, Chelonia mydas - PMID: 36749728
    * Lagopus muta - PMID: 37141262
    * Rissa tridactyla - PMID: 37590950
    * Trichomycterus rosablanca - PMID: 38513109
    * Pseudophryne corroboree - PMID: 40756765
    * Choloepus didactylus - https://www.biorxiv.org/content/10.1101/2025.09.25.678567v1
    * Hirundo rustica - PMID: 36662619
* in general it was difficult to annotate the dev stage for the animals with an exact age because there is not a lot of data on the distinction between neonate/juvenile/sexual maturity, i tried to be conservative about the annotations
* for replicates 
    * there are some libraries that share SRS/SAMN IDs, however i was able to confirm in the associated literature that they are not replicates
    * there were a few examples of replicates which were marked

### annotation summary

In [21]:
anat_summary = library_to_add[['infoOrgan', 'anatId', 'anatName', 'anatAnnotationStatus']]
unique_anat = anat_summary.drop_duplicates()
display_df(unique_anat)

,infoOrgan,anatId,anatName,anatAnnotationStatus
0,Brain,UBERON:0000955,brain,perfect match
1,Liver,UBERON:0002107,liver,perfect match
2,Muscle,UBERON:0002385,muscle tissue,perfect match
3,Testis,UBERON:0000473,testis,perfect match
4,Ovary,UBERON:0000992,ovary,perfect match
6,Spleen,UBERON:0002106,spleen,perfect match
7,Thymus,UBERON:0002370,thymus,perfect match
8,Gonads,UBERON:0000473,testis,perfect match
10,Eye,UBERON:0000970,eye,perfect match
14,Gonads,UBERON:0000991,gonad,perfect match


In [22]:
dev_summary = library_to_add[['infoStage', 'stageId', 'stageName', 'stageAnnotationStatus']]
unique_dev = dev_summary.drop_duplicates()
display_df(unique_dev)

,infoStage,stageId,stageName,stageAnnotationStatus
0,adult,UBERON:0000113,post-juvenile adult stage,perfect match
5,17 years,UBERON:0000112,sexually immature stage,other
11,NA,UBERON:0000104,life cycle,other
23,25 days,UBERON:0000112,sexually immature stage,other
24,4 months,UBERON:0000112,sexually immature stage,other
26,calf,UBERON:0000112,sexually immature stage,other
39,subadult,UBERON:0000112,sexually immature stage,perfect match
68,3 days,UBERON:0007221,neonate stage,other
70,Adult,UBERON:0000113,post-juvenile adult stage,perfect match
73,Embryos - 1 day post hatch,UBERON:0000069,larval stage,other


### set variables, import packages, define functions

In [1]:
experiment_id = "SRP184741"

path_to_create_exp_script = "/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py" 
experiment_type = "bulk"

path_to_output_main = "/Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/" 
path_to_output = "{}{}/".format(path_to_output_main, experiment_id)
library_path_from_script = "{}RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_path_from_script = "{}RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
library_to_add_path = "{}complete_RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_to_add_path = "{}complete_RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
script_file = "{}.ipynb".format(experiment_id)
commit_message_exp = '"adding annotated bulk experiment {}"'.format(experiment_id)
commit_message_py = '"adding annotation files for {} to notebook folder"'.format(experiment_id)


## to add to git
path_to_git_annotations = "/Users/scarsana/Desktop/git/expression-annotations/RNA_Seq/"
git_library_path = "{}RNASeqLibrary.tsv".format(path_to_git_annotations)
git_experiment_path = "{}RNASeqExperiment.tsv".format(path_to_git_annotations)

## validation
path_to_v_script = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/validate_annotations.py'
path_to_rules = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/rules/'
val_output = "{}{}/validation.tsv".format(path_to_output_main, experiment_id)

library_cols = ['#libraryId', 'experimentId', 'platform', 'SRSId', 'anatId', 'anatName', 'stageId', 'stageName', 'url_GSM', 'infoOrgan', 'infoStage', 'anatAnnotationStatus', 'anatBiologicalStatus', 'stageAnnotationStatus', 'sex', 'strain', 'genotype', 'speciesId', 'protocol', 'protocolType', 'RNASelection', 'globin_reduction', 'replicate', 'lib_name', 'sampleName', 'sampleAge_value', 'sampleAge_unit', 'PATOid', 'PATOname','EFOid', 'EFOname','comment', 'condition', 'physiologicalStatus', 'annotatorId', 'lastModificationDate']

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import os
import csv

# displays df with the scrollbar next to the DataFrame
def display_df(df):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    display(HTML("<div style='height: 300px; overflow: auto; width: fit-content'>" +
        df.style.to_html(index=False) + "</div>"))

# function that compares two columns in a dataframe and tells you which ones are not equal (case insensitive)
def compare_columns(df, col1, col2, return_col):
    compare_return = df[col1].str.lower() != df[col2].str.lower()  
    df.loc[compare_return, return_col] 
    if not any(compare_return):
        print("The two columns are equal (case insensitive)")
    else:
        print("The following rows are not equal: ")
        print(df.loc[compare_return, return_col])

# fixes formatting of file to match libreoffice settings/historic file format
def update_format(path):
    with open(path, 'r') as file:
        filedata = file.read()
    # Replace the target string
    filedata = filedata.replace("\t\"\"", "\t")
    # Write the file out again
    with open(path, 'w') as file:
        file.write(filedata)

# checks for duplicate values in a specific column and prints those values + the corresponding library id
def dup_check(df, column):
    duplicateCheck = df.duplicated(subset=[column], keep=False)
    duplicateCheck.sort_values(inplace=True)
    if duplicateCheck.unique().any() == False:
        print("no duplicate values in " + column)
    elif duplicateCheck.unique().any() == True and column != '#libraryId':
        dups = df[duplicateCheck].loc[:,['#libraryId', column]]
        df_dups = pd.DataFrame(dups)
        df_dups.sort_values(inplace=True, by=column)
        print(df_dups)
    elif duplicateCheck.unique().any() == True and column == '#libraryId':
        print(df[duplicateCheck].loc[:,['#libraryId']])

# prints all unique values in a specific column
def unique_sorted(df, column):
    unique = df[column].unique()
    unique.sort()
    print(unique)

### script

In [4]:
! python3 $path_to_create_exp_script $experiment_id $path_to_output $experiment_type

/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:120: SyntaxWarning: invalid escape sequence '\('
  all_protoc = [w.replace('(', '\(') for w in all_protoc]
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:121: SyntaxWarning: invalid escape sequence '\)'
  all_protoc = [w.replace(')', '\)') for w in all_protoc] 
Be patient, it may take a few minutes.
100%|█████████████████████████████████████████| 175/175 [02:58<00:00,  1.02s/it]
0 samples dont have attributes, try to find them somewhere else
0it [00:00, ?it/s]
0 samples dont have attributes


### library annnotations

In [5]:
library = pd.read_csv(library_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX10857540,SRP184741,Illumina NovaSeq 6000,SRS8953033,UBERON:0000955,brain,UBERON:0000113,post-juvenile adult stage,,Brain,adult,perfect match,not documented,perfect match,M,,,34773,,,polyA,,,Brain sample of Alosa sapidissima,SAMN19117395,,,,,,,PMID: 39786563,Flash Frozen,,,15/04/2026,mRNA seq for annotation,,Brain_fAloSap1,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
1,SRX10857541,SRP184741,Illumina NovaSeq 6000,SRS8953509,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,Liver,adult,perfect match,not documented,perfect match,M,,,34773,,,polyA,,,Livery sample of Alosa sapidissima,SAMN19117396,,,,,,,PMID: 39786563,Flash Frozen,,,15/04/2026,mRNA seq for annotation,,Liver_fAloSap1,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
2,SRX10857542,SRP184741,Illumina NovaSeq 6000,SRS8953034,UBERON:0002385,muscle tissue,UBERON:0000113,post-juvenile adult stage,,Muscle,adult,perfect match,not documented,perfect match,M,,,34773,,,polyA,,,Muscle sample of Alosa sapidissima,SAMN19117397,,,,,,,PMID: 39786563,Flash Frozen,,,15/04/2026,mRNA seq for annotation,,Muscle_fAloSap1,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
3,SRX10857543,SRP184741,Illumina NovaSeq 6000,SRS8953035,UBERON:0000473,testis,UBERON:0000113,post-juvenile adult stage,,Testis,adult,perfect match,not documented,perfect match,M,,,34773,,,polyA,,,Testis sample of Alosa sapidissima,SAMN19117398,,,,,,,PMID: 39786563,Flash Frozen,,,15/04/2026,mRNA seq for annotation,,Testis_fAloSap1,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
4,SRX10857544,SRP184741,Illumina NovaSeq 6000,SRS8953036,UBERON:0000992,ovary,UBERON:0000113,post-juvenile adult stage,,Ovary,adult,perfect match,not documented,perfect match,F,,,34773,,,polyA,,,Ovary sample of Alosa sapidissima,SAMN19117399,,,,,,,PMID: 39786563,Flash Frozen,,,15/04/2026,mRNA seq for annotation,,Ovary_fAloSap2,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
5,SRX10863130,SRP184741,Illumina NovaSeq 6000,SRS8958491,UBERON:0000955,brain,UBERON:0000112,sexually immature stage,,Brain,17 years,perfect match,not documented,other,M,,,8469,TruSeq Stranded mRNA,full_length,polyA,,,Brain sample of Chelonia mydas,SAMN19133328,17,year,,,,,PMID: 36749728,frozen,,,15/04/2026,mRNA seq for annotation,,Brain_rCheMyd1,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
6,SRX10863131,SRP184741,Illumina NovaSeq 6000,SRS8958492,UBERON:0002106,spleen,UBERON:0000112,sexually immature stage,,Spleen,17 years,perfect match,not documented,other,M,,,8469,TruSeq Stranded mRNA,full_length,polyA,,,Spleen sample of Chelonia mydas,SAMN19133329,17,year,,,,,PMID: 36749728,frozen,,,15/04/2026,mRNA seq for annotation,,Spleen_rCheMyd1,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
7,SRX10863132,SRP184741,Illumina NovaSeq 6000,SRS8958493,UBERON:0002370,thymus,UBERON:0000112,sexually immature stage,,Thymus,17 years,perfect match,not documented,other,M,,,8469,TruSeq Stranded mRNA,full_length,polyA,,,Thymus sample of Chelonia mydas,SAMN19133330,17,year,,,,,PMID: 36749728,frozen,,,15/04/2026,mRNA seq for annotation,,Thymus_rCheMyd1,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
8,SRX10863133,SRP184741,Illumina NovaSeq 6000,SRS8958494,UBERON:0000473,testis,UBERON:0000112,sexually immature stage,,Gonads,17 years,perfect match,not documented,other,M,,,8469,TruSeq Stranded mRNA,full_length,polyA,,,Gonads sample of Chelonia mydas,SAMN19133331,17,year,,,,,PMID: 36749728,frozen,,,15/04/2026,mRNA seq for annotation,,Gonads_rCheMyd1,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
9,SRX10895064,SRP184741,Illumina NovaSeq 6000,SRS8987850,UBERON:0000955,br

#### anatomical entity
* [uberon ols](https://www.ebi.ac.uk/ols4/ontologies/uberon)

In [6]:
unique_sorted(library, "infoOrgan")

['Blood' 'Brain' 'Embryos (1 day post hatch)' 'Eye' 'Eyes' 'Fins' 'Gonads'
 'Jaws' 'Kidney' 'Liver' 'Lung' 'Muscle' 'Ovary' 'Spleen' 'Testis'
 'Thymus' 'blood' 'brain' 'gillRaker' 'head' 'heart' 'heart muscle'
 'kidney' 'liver' 'lung' 'muscle' 'ovaries' 'ovary' 'pectorialis muscle'
 'skin' 'spleen' 'testes' 'whole body']


#### stage
- [species specific developmental ontologies](https://github.com/obophenotype/developmental-stage-ontologies/tree/master/src/ontology/components)

In [7]:
unique_sorted(library, "infoStage")

['17 years' '25 days' '3 days' '4 months' 'Adult'
 'Embryos - 1 day post hatch' 'Hatchling' 'NA' 'adult' 'calf' 'subadult']


#### sex, strain, genotype, speciesId
- uniprot [strain list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/strains)
- uniprot [species list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/speclist)
- bgee [strain mapping](https://gitlab.sib.swiss/Bgee/expression-annotations/-/tree/develop/Strains?ref_type=heads)

In [ ]:
#library.loc[:,'sex'] = ''
#library.loc[library["sex"] == "male", "sex"] = "M"
#library.loc[library["sex"] == "female", "sex"] = "F"

#library.loc[:,'strain'] = ''

#library.loc[:,'genotype'] = ''

#library.loc[:,'speciesId'] = ''

# view
display_df(library)

#### protocol
see [bulk kits](https://gitlab.sib.swiss/Bgee/scRNA-Seq/-/blob/main/scripts/bulk_kits.csv) for some common protocols

In [8]:
# making these variables because we use them again in the experiment file
my_protocol = 'TruSeq Stranded mRNA,NEBNext Ultra RNA Library Prep Kit,NEBNext Ultra (II) RNA Library Prep Kit,Illumina Stranded mRNA Prep'
# full_length or 3'
my_protocolType = 'full_length'

#library.loc[:,'protocol'] = my_protocol
#library.loc[:,'protocolType'] = my_protocolType
# polyA, ribo-minus, miRNA, lncRNA, circRNA
#library.loc[:,'RNASelection'] = ''

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX10857540,SRP184741,Illumina NovaSeq 6000,SRS8953033,UBERON:0000955,brain,UBERON:0000113,post-juvenile adult stage,,Brain,adult,perfect match,not documented,perfect match,M,,,34773,,,polyA,,,Brain sample of Alosa sapidissima,SAMN19117395,,,,,,,PMID: 39786563,Flash Frozen,,,15/04/2026,mRNA seq for annotation,,Brain_fAloSap1,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
1,SRX10857541,SRP184741,Illumina NovaSeq 6000,SRS8953509,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,Liver,adult,perfect match,not documented,perfect match,M,,,34773,,,polyA,,,Livery sample of Alosa sapidissima,SAMN19117396,,,,,,,PMID: 39786563,Flash Frozen,,,15/04/2026,mRNA seq for annotation,,Liver_fAloSap1,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
2,SRX10857542,SRP184741,Illumina NovaSeq 6000,SRS8953034,UBERON:0002385,muscle tissue,UBERON:0000113,post-juvenile adult stage,,Muscle,adult,perfect match,not documented,perfect match,M,,,34773,,,polyA,,,Muscle sample of Alosa sapidissima,SAMN19117397,,,,,,,PMID: 39786563,Flash Frozen,,,15/04/2026,mRNA seq for annotation,,Muscle_fAloSap1,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
3,SRX10857543,SRP184741,Illumina NovaSeq 6000,SRS8953035,UBERON:0000473,testis,UBERON:0000113,post-juvenile adult stage,,Testis,adult,perfect match,not documented,perfect match,M,,,34773,,,polyA,,,Testis sample of Alosa sapidissima,SAMN19117398,,,,,,,PMID: 39786563,Flash Frozen,,,15/04/2026,mRNA seq for annotation,,Testis_fAloSap1,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
4,SRX10857544,SRP184741,Illumina NovaSeq 6000,SRS8953036,UBERON:0000992,ovary,UBERON:0000113,post-juvenile adult stage,,Ovary,adult,perfect match,not documented,perfect match,F,,,34773,,,polyA,,,Ovary sample of Alosa sapidissima,SAMN19117399,,,,,,,PMID: 39786563,Flash Frozen,,,15/04/2026,mRNA seq for annotation,,Ovary_fAloSap2,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
5,SRX10863130,SRP184741,Illumina NovaSeq 6000,SRS8958491,UBERON:0000955,brain,UBERON:0000112,sexually immature stage,,Brain,17 years,perfect match,not documented,other,M,,,8469,TruSeq Stranded mRNA,full_length,polyA,,,Brain sample of Chelonia mydas,SAMN19133328,17,year,,,,,PMID: 36749728,frozen,,,15/04/2026,mRNA seq for annotation,,Brain_rCheMyd1,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
6,SRX10863131,SRP184741,Illumina NovaSeq 6000,SRS8958492,UBERON:0002106,spleen,UBERON:0000112,sexually immature stage,,Spleen,17 years,perfect match,not documented,other,M,,,8469,TruSeq Stranded mRNA,full_length,polyA,,,Spleen sample of Chelonia mydas,SAMN19133329,17,year,,,,,PMID: 36749728,frozen,,,15/04/2026,mRNA seq for annotation,,Spleen_rCheMyd1,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
7,SRX10863132,SRP184741,Illumina NovaSeq 6000,SRS8958493,UBERON:0002370,thymus,UBERON:0000112,sexually immature stage,,Thymus,17 years,perfect match,not documented,other,M,,,8469,TruSeq Stranded mRNA,full_length,polyA,,,Thymus sample of Chelonia mydas,SAMN19133330,17,year,,,,,PMID: 36749728,frozen,,,15/04/2026,mRNA seq for annotation,,Thymus_rCheMyd1,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
8,SRX10863133,SRP184741,Illumina NovaSeq 6000,SRS8958494,UBERON:0000473,testis,UBERON:0000112,sexually immature stage,,Gonads,17 years,perfect match,not documented,other,M,,,8469,TruSeq Stranded mRNA,full_length,polyA,,,Gonads sample of Chelonia mydas,SAMN19133331,17,year,,,,,PMID: 36749728,frozen,,,15/04/2026,mRNA seq for annotation,,Gonads_rCheMyd1,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
9,SRX10895064,SRP184741,Illumina NovaSeq 6000,SRS8987850,UBERON:0000955,br

#### globin, replicates

In [9]:
# check for duplicate SRSId values
dup_check(library, "SRSId")

     #libraryId        SRSId
24  SRX21609206  SRS18706220
51  SRX23382447  SRS18706220
27  SRX21609210  SRS18782498
28  SRX21609211  SRS18782498
43  SRX22284545  SRS19335842
41  SRX22284543  SRS19335842
44  SRX22284546  SRS19335842
29  SRX22284531  SRS19335842
30  SRX22284532  SRS19335842
48  SRX22284550  SRS19335844
58  SRX23426436  SRS19335844
57  SRX23426435  SRS19335844
47  SRX22284549  SRS19335844
31  SRX22284533  SRS19335844
34  SRX22284536  SRS19335845
35  SRX22284537  SRS19335845
36  SRX22284538  SRS19335845
45  SRX22284547  SRS19335847
46  SRX22284548  SRS19335847
38  SRX22284540  SRS19335847
37  SRX22284539  SRS19335847
40  SRX22284542  SRS19335848
39  SRX22284541  SRS19335848
42  SRX22284544  SRS19335848
55  SRX23408951  SRS20264490
53  SRX23408949  SRS20264490
54  SRX23408950  SRS20264490
56  SRX23408952  SRS20264490
60   SRX5353344   SRS4344598
63   SRX5491149   SRS4344598
65   SRX5491153   SRS4344601
61   SRX5353350   SRS4344601
62   SRX5353352   SRS4344603
64   SRX549115

/var/folders/b5/crkp117d43q5mcndnwlrww3w0000gn/T/ipykernel_70243/3311601719.py:43: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  dups = df[duplicateCheck].loc[:,['#libraryId', column]]


In [ ]:
#library.loc[:,'globin_reduction'] = 'Y'

# replicates
#library.loc[library["#libraryId"] == "old", "replicate"] = "1"
#library.loc[library["#libraryId"].isin(["one", "two"]), "replicate"] = "1"

# view
display_df(library)

#### sample age, pato, physiological status
* [PATO](https://www.ebi.ac.uk/ols4/ontologies/pato)
* [EFO](https://www.ebi.ac.uk/ols4/ontologies/efo)

In [ ]:
#library.loc[:,'sampleAge_value'] = ''
#library.loc[:,'sampleAge_unit'] = ''

# ex. castrated male
#library.loc[:,'PATOid'] = ''
#library.loc[:,'PATOname'] = ''

# ex. castrated, pregnant, pre-smoltification, post-smoltification, laying eggs
#library.loc[:,'physiologicalStatus'] = ''

# ex. left, right
#library.loc[:,'EFOid'] = ''
#library.loc[:,'EFOname'] = ''

# view
display_df(library)

#### condition

In [ ]:
# ex. control, diet, light, reproductive capacity, time post mortem, time post feeding, 
# exercise details, menstruation, personality, litter size 
#library.loc[library["condition"] == "old", "condition"] = "new"

# view
display_df(library)

#### annotator id, last modification date

In [10]:
library.loc[:,'annotatorId'] = 'SAC'
library.loc[:,'lastModificationDate'] = '2026-04-16'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX10857540,SRP184741,Illumina NovaSeq 6000,SRS8953033,UBERON:0000955,brain,UBERON:0000113,post-juvenile adult stage,,Brain,adult,perfect match,not documented,perfect match,M,,,34773,,,polyA,,,Brain sample of Alosa sapidissima,SAMN19117395,,,,,,,PMID: 39786563,Flash Frozen,,SAC,2026-04-16,mRNA seq for annotation,,Brain_fAloSap1,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
1,SRX10857541,SRP184741,Illumina NovaSeq 6000,SRS8953509,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,Liver,adult,perfect match,not documented,perfect match,M,,,34773,,,polyA,,,Livery sample of Alosa sapidissima,SAMN19117396,,,,,,,PMID: 39786563,Flash Frozen,,SAC,2026-04-16,mRNA seq for annotation,,Liver_fAloSap1,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
2,SRX10857542,SRP184741,Illumina NovaSeq 6000,SRS8953034,UBERON:0002385,muscle tissue,UBERON:0000113,post-juvenile adult stage,,Muscle,adult,perfect match,not documented,perfect match,M,,,34773,,,polyA,,,Muscle sample of Alosa sapidissima,SAMN19117397,,,,,,,PMID: 39786563,Flash Frozen,,SAC,2026-04-16,mRNA seq for annotation,,Muscle_fAloSap1,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
3,SRX10857543,SRP184741,Illumina NovaSeq 6000,SRS8953035,UBERON:0000473,testis,UBERON:0000113,post-juvenile adult stage,,Testis,adult,perfect match,not documented,perfect match,M,,,34773,,,polyA,,,Testis sample of Alosa sapidissima,SAMN19117398,,,,,,,PMID: 39786563,Flash Frozen,,SAC,2026-04-16,mRNA seq for annotation,,Testis_fAloSap1,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
4,SRX10857544,SRP184741,Illumina NovaSeq 6000,SRS8953036,UBERON:0000992,ovary,UBERON:0000113,post-juvenile adult stage,,Ovary,adult,perfect match,not documented,perfect match,F,,,34773,,,polyA,,,Ovary sample of Alosa sapidissima,SAMN19117399,,,,,,,PMID: 39786563,Flash Frozen,,SAC,2026-04-16,mRNA seq for annotation,,Ovary_fAloSap2,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
5,SRX10863130,SRP184741,Illumina NovaSeq 6000,SRS8958491,UBERON:0000955,brain,UBERON:0000112,sexually immature stage,,Brain,17 years,perfect match,not documented,other,M,,,8469,TruSeq Stranded mRNA,full_length,polyA,,,Brain sample of Chelonia mydas,SAMN19133328,17,year,,,,,PMID: 36749728,frozen,,SAC,2026-04-16,mRNA seq for annotation,,Brain_rCheMyd1,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
6,SRX10863131,SRP184741,Illumina NovaSeq 6000,SRS8958492,UBERON:0002106,spleen,UBERON:0000112,sexually immature stage,,Spleen,17 years,perfect match,not documented,other,M,,,8469,TruSeq Stranded mRNA,full_length,polyA,,,Spleen sample of Chelonia mydas,SAMN19133329,17,year,,,,,PMID: 36749728,frozen,,SAC,2026-04-16,mRNA seq for annotation,,Spleen_rCheMyd1,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
7,SRX10863132,SRP184741,Illumina NovaSeq 6000,SRS8958493,UBERON:0002370,thymus,UBERON:0000112,sexually immature stage,,Thymus,17 years,perfect match,not documented,other,M,,,8469,TruSeq Stranded mRNA,full_length,polyA,,,Thymus sample of Chelonia mydas,SAMN19133330,17,year,,,,,PMID: 36749728,frozen,,SAC,2026-04-16,mRNA seq for annotation,,Thymus_rCheMyd1,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
8,SRX10863133,SRP184741,Illumina NovaSeq 6000,SRS8958494,UBERON:0000473,testis,UBERON:0000112,sexually immature stage,,Gonads,17 years,perfect match,not documented,other,M,,,8469,TruSeq Stranded mRNA,full_length,polyA,,,Gonads sample of Chelonia mydas,SAMN19133331,17,year,,,,,PMID: 36749728,frozen,,SAC,2026-04-16,mRNA seq for annotation,,Gonads_rCheMyd1,,,,adult,,TRANSCRIPTOMIC,Oligo-dT
9,SRX10895064,SRP184741,Illumina NovaSeq 6000,S

#### comments

In [ ]:
#library.loc[:,'comment'] = ''

#### save complete file with correct columns

In [11]:
library_file_complete = library[library_cols]
library_file_complete.to_csv(library_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

# view
display_df(library_file_complete)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX10857540,SRP184741,Illumina NovaSeq 6000,SRS8953033,UBERON:0000955,brain,UBERON:0000113,post-juvenile adult stage,,Brain,adult,perfect match,not documented,perfect match,M,,,34773,,,polyA,,,Brain sample of Alosa sapidissima,SAMN19117395,,,,,,,PMID: 39786563,Flash Frozen,,SAC,2026-04-16
1,SRX10857541,SRP184741,Illumina NovaSeq 6000,SRS8953509,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,Liver,adult,perfect match,not documented,perfect match,M,,,34773,,,polyA,,,Livery sample of Alosa sapidissima,SAMN19117396,,,,,,,PMID: 39786563,Flash Frozen,,SAC,2026-04-16
2,SRX10857542,SRP184741,Illumina NovaSeq 6000,SRS8953034,UBERON:0002385,muscle tissue,UBERON:0000113,post-juvenile adult stage,,Muscle,adult,perfect match,not documented,perfect match,M,,,34773,,,polyA,,,Muscle sample of Alosa sapidissima,SAMN19117397,,,,,,,PMID: 39786563,Flash Frozen,,SAC,2026-04-16
3,SRX10857543,SRP184741,Illumina NovaSeq 6000,SRS8953035,UBERON:0000473,testis,UBERON:0000113,post-juvenile adult stage,,Testis,adult,perfect match,not documented,perfect match,M,,,34773,,,polyA,,,Testis sample of Alosa sapidissima,SAMN19117398,,,,,,,PMID: 39786563,Flash Frozen,,SAC,2026-04-16
4,SRX10857544,SRP184741,Illumina NovaSeq 6000,SRS8953036,UBERON:0000992,ovary,UBERON:0000113,post-juvenile adult stage,,Ovary,adult,perfect match,not documented,perfect match,F,,,34773,,,polyA,,,Ovary sample of Alosa sapidissima,SAMN19117399,,,,,,,PMID: 39786563,Flash Frozen,,SAC,2026-04-16
5,SRX10863130,SRP184741,Illumina NovaSeq 6000,SRS8958491,UBERON:0000955,brain,UBERON:0000112,sexually immature stage,,Brain,17 years,perfect match,not documented,other,M,,,8469,TruSeq Stranded mRNA,full_length,polyA,,,Brain sample of Chelonia mydas,SAMN19133328,17,year,,,,,PMID: 36749728,frozen,,SAC,2026-04-16
6,SRX10863131,SRP184741,Illumina NovaSeq 6000,SRS8958492,UBERON:0002106,spleen,UBERON:0000112,sexually immature stage,,Spleen,17 years,perfect match,not documented,other,M,,,8469,TruSeq Stranded mRNA,full_length,polyA,,,Spleen sample of Chelonia mydas,SAMN19133329,17,year,,,,,PMID: 36749728,frozen,,SAC,2026-04-16
7,SRX10863132,SRP184741,Illumina NovaSeq 6000,SRS8958493,UBERON:0002370,thymus,UBERON:0000112,sexually immature stage,,Thymus,17 years,perfect match,not documented,other,M,,,8469,TruSeq Stranded mRNA,full_length,polyA,,,Thymus sample of Chelonia mydas,SAMN19133330,17,year,,,,,PMID: 36749728,frozen,,SAC,2026-04-16
8,SRX10863133,SRP184741,Illumina NovaSeq 6000,SRS8958494,UBERON:0000473,testis,UBERON:0000112,sexually immature stage,,Gonads,17 years,perfect match,not documented,other,M,,,8469,TruSeq Stranded mRNA,full_length,polyA,,,Gonads sample of Chelonia mydas,SAMN19133331,17,year,,,,,PMID: 36749728,frozen,,SAC,2026-04-16
9,SRX10895064,SRP184741,Illumina NovaSeq 6000,SRS8987850,UBERON:0000955,brain,UBERON:0000113,post-juvenile adult stage,,Brain,adult,perfect match,not documented,perfect match,NA,,,72105,,,polyA,,,Brain sample of Sebastes umbrosus,SAMN19190429,,,,,,,,Flash Frozen,,SAC,2026-04-16


### experiment annotations

In [12]:
experiment = pd.read_csv(experiment_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP184741,G10K_VGP_Transcriptome_Sequencing,"The Vertebrate Genomes Project aims to provide annotated reference genome sequences for all vertebrate species. This project contains transcriptome data from various sequencing platforms to support transcript and gene annotation of reference genomes generated by VGP. The raw data are currently under a G10K-VGP publication embargo until removed from this description, following the G10K data use policy at the following URL: https://genome10k.soe.ucsc.edu/about/data_use_policy.",SRA,,,,Illumina Stranded mRNA Prep,full_length,,PRJNA516733,36344967,,10.1186/s12915-022-01427-8,,


#### experiment and protocol details

In [13]:
# this will give you the number of rows in the complete library file 
# this should be the number of annotated libraries
ann_lib = len(library_file_complete.index)
len(library_file_complete.index)

100

In [15]:
# partial or total
experiment.loc[:,'experimentStatus'] = 'partial'
# Bgee 1K
experiment.loc[:,'projectTags'] = 'Bgee 1K' 
# see above cell, also can add as free text
experiment.loc[:,'numberOfAnnotatedLibraries'] = ann_lib

# these variables should already exist from above but if not can just add as free text
experiment.loc[:,'protocol'] = my_protocol
experiment.loc[:,'protocolType'] = my_protocolType

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP184741,G10K_VGP_Transcriptome_Sequencing,"The Vertebrate Genomes Project aims to provide annotated reference genome sequences for all vertebrate species. This project contains transcriptome data from various sequencing platforms to support transcript and gene annotation of reference genomes generated by VGP. The raw data are currently under a G10K-VGP publication embargo until removed from this description, following the G10K data use policy at the following URL: https://genome10k.soe.ucsc.edu/about/data_use_policy.",SRA,partial,Bgee 1K,100,"TruSeq Stranded mRNA,NEBNext Ultra RNA Library Prep Kit,NEBNext Ultra (II) RNA Library Prep Kit,Illumina Stranded mRNA Prep",full_length,,PRJNA516733,36344967,,10.1186/s12915-022-01427-8,,


#### paper and xrefs

In [16]:
#experiment.loc[:,'GSE'] = ''
#experiment.loc[:,'Bioproject'] = '' 
experiment.loc[:,'PMID'] = '33911273'
experiment.loc[:,'reference_url'] = 'https://genome10k.ucsc.edu/'
experiment.loc[:,'DOI'] = '10.1038/s41586-021-03451-0'
experiment.loc[:,'xrefs'] = '33305470[PMID],39786563[PMID],36749728[PMID],37141262[PMID],37590950[PMID],38513109[PMID],40756765[PMID],36662619[PMID]'

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP184741,G10K_VGP_Transcriptome_Sequencing,"The Vertebrate Genomes Project aims to provide annotated reference genome sequences for all vertebrate species. This project contains transcriptome data from various sequencing platforms to support transcript and gene annotation of reference genomes generated by VGP. The raw data are currently under a G10K-VGP publication embargo until removed from this description, following the G10K data use policy at the following URL: https://genome10k.soe.ucsc.edu/about/data_use_policy.",SRA,partial,Bgee 1K,100,"TruSeq Stranded mRNA,NEBNext Ultra RNA Library Prep Kit,NEBNext Ultra (II) RNA Library Prep Kit,Illumina Stranded mRNA Prep",full_length,,PRJNA516733,33911273,https://genome10k.ucsc.edu/,10.1038/s41586-021-03451-0,"33305470[PMID],39786563[PMID],36749728[PMID],37141262[PMID],37590950[PMID],38513109[PMID],40756765[PMID],36662619[PMID]",


#### comments

In [17]:
experiment.loc[:,'comment'] = 'not sure exactly what paper or DOI to put as the primary, the data is described in separate papers. including PMID: 33305470, PMID: 39786563, PMID: 36749728, PMID: 37141262, PMID: 37590950, PMID: 38513109, PMID: 40756765, PMID: 36662619, https://www.biorxiv.org/content/10.1101/2025.09.25.678567v1. i used the recent paper describing the vertebrate genome project as the main PMID/DOI'

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP184741,G10K_VGP_Transcriptome_Sequencing,"The Vertebrate Genomes Project aims to provide annotated reference genome sequences for all vertebrate species. This project contains transcriptome data from various sequencing platforms to support transcript and gene annotation of reference genomes generated by VGP. The raw data are currently under a G10K-VGP publication embargo until removed from this description, following the G10K data use policy at the following URL: https://genome10k.soe.ucsc.edu/about/data_use_policy.",SRA,partial,Bgee 1K,100,"TruSeq Stranded mRNA,NEBNext Ultra RNA Library Prep Kit,NEBNext Ultra (II) RNA Library Prep Kit,Illumina Stranded mRNA Prep",full_length,,PRJNA516733,33911273,https://genome10k.ucsc.edu/,10.1038/s41586-021-03451-0,"33305470[PMID],39786563[PMID],36749728[PMID],37141262[PMID],37590950[PMID],38513109[PMID],40756765[PMID],36662619[PMID]","not sure exactly what paper or DOI to put as the primary, the data is described in separate papers. including PMID: 33305470, PMID: 39786563, PMID: 36749728, PMID: 37141262, PMID: 37590950, PMID: 38513109, PMID: 40756765, PMID: 36662619, https://www.biorxiv.org/content/10.1101/2025.09.25.678567v1. i used the recent paper describing the vertebrate genome project as the main PMID/DOI"


#### save complete file

In [18]:
experiment.to_csv(experiment_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

### QA time

In [19]:
library_to_add = pd.read_csv(library_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
experiment_to_add = pd.read_csv(experiment_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

In [20]:
! python3 $path_to_v_script --bulk-exp $experiment_to_add_path --bulk-lib $library_to_add_path --rules-dir $path_to_rules --out $val_output --strict

Total issues: 1
Errors: 1
Warnings: 0
Top codes:
  - BAD_VALUE: 1


#### check columns match

In [23]:
# pull from git and pull in library/experiment file
! git pull
git_library = pd.read_csv(git_library_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
git_experiment = pd.read_csv(git_experiment_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

# library file
if set(library_to_add.columns) == set(git_library.columns):
    print('The columns in the library file match')
else:
    print('The columns in the library file DO NOT MATCH')

# experiment file
if set(experiment_to_add.columns) == set(git_experiment.columns):
    print('The columns in the experiment file match')
else:
    print('The columns in the experiment file DO NOT MATCH')


# maybe to make this something more like "COLUMNS GOOD - LIBRARY" and "COLUMNS BAD - EXPERIMENT"

Already up to date.
The columns in the library file match
The columns in the experiment file match


#### view files

In [24]:
library_git_plus_new = pd.concat([git_library, library_to_add], ignore_index = True, sort = False)
old_length = git_library.shape[0]
start = old_length - 2
end = old_length + 5
view_lib = library_git_plus_new.iloc[start:end]
view_lib

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
61868,SRX9474695,SRP292185,HiSeq X Ten,SRS7684131,UBERON:8480071,skin of lobe of tail,UBERON:0000113,post-juvenile adult stage,,"skin from tail fluke, dermis and epidermis",adult,perfect match,not documented,perfect match,F,,,9755,,,polyA,,,Model organism or animal sample from Physeter ...,SAMN16752490,,,,,,,"PMID: 33562637, additional metadata from PMID:...",pregnant,,SAC,2026-04-15
61869,SRX5087289,SRP171999,Illumina HiSeq 4000,SRS4099392,UBERON:0000014,zone of skin,UBERON:0000113,post-juvenile adult stage,,Skin,Adult,perfect match,not documented,perfect match,NA,,,9771,,,polyA,,,B.musculus_Skin,SAMN10511405,,,,,,,PMID: 30895322,,,SAC,2026-04-15
61870,SRX10857540,SRP184741,Illumina NovaSeq 6000,SRS8953033,UBERON:0000955,brain,UBERON:0000113,post-juvenile adult stage,,Brain,adult,perfect match,not documented,perfect match,M,,,34773,,,polyA,,,Brain sample of Alosa sapidissima,SAMN19117395,,,,,,,PMID: 39786563,Flash Frozen,,SAC,2026-04-16
61871,SRX10857541,SRP184741,Illumina NovaSeq 6000,SRS8953509,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,Liver,adult,perfect match,not documented,perfect match,M,,,34773,,,polyA,,,Livery sample of Alosa sapidissima,SAMN19117396,,,,,,,PMID: 39786563,Flash Frozen,,SAC,2026-04-16
61872,SRX10857542,SRP184741,Illumina NovaSeq 6000,SRS8953034,UBERON:0002385,muscle tissue,UBERON:0000113,post-juvenile adult stage,,Muscle,adult,perfect match,not documented,perfect match,M,,,34773,,,polyA,,,Muscle sample of Alosa sapidissima,SAMN19117397,,,,,,,PMID: 39786563,Flash Frozen,,SAC,2026-04-16
61873,SRX10857543,SRP184741,Illumina NovaSeq 6000,SRS8953035,UBERON:0000473,testis,UBERON:0000113,post-juvenile adult stage,,Testis,adult,perfect match,not documented,perfect match,M,,,34773,,,polyA,,,Testis sample of Alosa sapidissima,SAMN19117398,,,,,,,PMID: 39786563,Flash Frozen,,SAC,2026-04-16
61874,SRX10857544,SRP184741,Illumina NovaSeq 6000,SRS8953036,UBERON:0000992,ovary,UBERON:0000113,post-juvenile adult stage,,Ovary,adult,perfect match,not documented,perfect match,F,,,34773,,,polyA,,,Ovary sample of Alosa sapidissima,SAMN19117399,,,,,,,PMID: 39786563,Flash Frozen,,SAC,2026-04-16


In [25]:
experiment_git_plus_new = pd.concat([git_experiment, experiment_to_add], ignore_index = True, sort = False)
experiment_git_plus_new.tail(n=3)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
1186,SRP292185,Integrated Full-length Transcriptome and RNA-S...,21 immune-related pathways were enriched by ge...,SRA,partial,Bgee 1K,1,,,,PRJNA676103,33562637,https://pmc.ncbi.nlm.nih.gov/articles/PMC7914425/,10.3390/genes12020233,31179986[PMID],removed pacbio library
1187,SRP171999,Balaenoptera musculus - Skin Transcriptome,De novo assembly and annotation of Balaenopter...,SRA,total,Bgee 1K,1,,,,PRJNA507895,30895322,https://pmc.ncbi.nlm.nih.gov/articles/PMC6526905/,10.1093/molbev/msz068,,
1188,SRP184741,G10K_VGP_Transcriptome_Sequencing,The Vertebrate Genomes Project aims to provide...,SRA,partial,Bgee 1K,100,"TruSeq Stranded mRNA,NEBNext Ultra RNA Library...",full_length,,PRJNA516733,33911273,https://genome10k.ucsc.edu/,10.1038/s41586-021-03451-0,"33305470[PMID],39786563[PMID],36749728[PMID],3...",not sure exactly what paper or DOI to put as t...


### add annotations to git

In [26]:
! git pull

Already up to date.


In [27]:
library_git_plus_new.to_csv(git_library_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
experiment_git_plus_new.to_csv(git_experiment_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
update_format(git_library_path)
update_format(git_experiment_path)

In [ ]:
! git status

In [28]:
! git add $git_experiment_path $git_library_path

In [29]:
! git commit -m $commit_message_exp

[develop 63a8a83] adding annotated bulk experiment SRP184741
 2 files changed, 101 insertions(+)


In [30]:
! git push

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 7.20 KiB | 1.80 MiB/s, done.
Total 5 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
remote: 
remote: To create a merge request for develop, visit:
remote:   https://gitlab.sib.swiss/Bgee/expression-annotations/-/merge_requests/new?merge_request%5Bsource_branch%5D=develop
remote: 
To https://gitlab.sib.swiss/Bgee/expression-annotations.git/
   b39e1f5..63a8a83  develop -> develop


### add annotation folder and script to git

In [ ]:
! git pull

1. run first two cells (annotation summary)
2. export as html

In [ ]:
! git add $path_to_output

In [ ]:
! git commit -m $commit_message_py

In [ ]:
! git push